# Task 6 — Historical Catalog Entry Analytics

Assignment: Content Success Analytics Engine (slide 11). The available dataset has no commercial outcome. This implementation predicts addition calendar year minus release year and produces reproducible catalog insights. It does not measure success, demand, or exact waiting time. Source snapshot ends September 25, 2021.

First-row entity deduplication leaves 8,783 titles. Fourteen negative differences are audited and excluded, leaving 8,769. Training is through 2019; chronological CV validates 2017/2018/2019; final test is 2020–2021. The selected model remains training-only. Encoders, scalers and log-target transforms fit within each fold. No year or addition date enters predictors.


In [1]:
from netflix_ml.tasks.task_06.service import load_analytics

service = load_analytics()
result = service.results
print('Run:', result['run_id'])
print('Audit:', result['audit'])
print('Selected:', result['model_id'], result['feature_set'])


Run: 20261106T131312231321Z
Audit: {'source_sha256': '90b02beeb554921f81d3b9a774d0f7766f6a65240162565f6b4082165dc2a1d3', 'raw_rows': 8790, 'entity_rows': 8783, 'eligible_rows': 8769, 'duplicate_mapping': [{'removed_id': 's5965', 'retained_id': 's3997'}, {'removed_id': 's5966', 'retained_id': 's4523'}, {'removed_id': 's5967', 'retained_id': 's3963'}, {'removed_id': 's6530', 'retained_id': 's3372'}, {'removed_id': 's6706', 'retained_id': 's304'}, {'removed_id': 's7346', 'retained_id': 's160'}, {'removed_id': 's8023', 'retained_id': 's1271'}], 'snapshot_date': '2021-09-25', 'anomalies': [{'show_id': 's5395', 'title': 'Hans Teeuwen: Real Rancour', 'type': 'Movie', 'release_year': 2018, 'date_added': '7/1/2017', 'lag_years': -1}, {'show_id': 's7064', 'title': 'Incoming', 'type': 'Movie', 'release_year': 2019, 'date_added': '10/26/2018', 'lag_years': -1}, {'show_id': 's7113', 'title': 'Jack Taylor', 'type': 'TV Show', 'release_year': 2016, 'date_added': '3/31/2013', 'lag_years': -3}, {'show_

In [2]:
import pandas as pd

comparison = pd.DataFrame([{'model':r['model'], 'features':r['feature_set'], 'cv_mae':r['cv_mae'], **r['test_metrics']} for r in result['comparison']])
print(comparison.to_string(index=False))
print('Baseline comparison:', result['quality'])
print(pd.DataFrame(result['errors']).to_string(index=False))
print(pd.DataFrame(result['importance']).to_string(index=False))


                  model features   cv_mae      mae     rmse  median_absolute_error        r2
          median_global baseline 4.113687 4.967991 9.778519               1.000000 -0.220753
            median_type baseline 4.196175 4.967991 9.778519               1.000000 -0.220753
         ridge_base_0.1     base 3.942910 4.449234 8.569977               1.640953  0.062350
           ridge_base_1     base 3.950094 4.428835 8.448927               1.645492  0.088651
          ridge_base_10     base 3.926413 4.326510 7.903836               1.657363  0.202451
         ridge_base_100     base 4.064663 4.586716 8.595192               1.612050  0.056824
       forest_base_10_2     base 3.957301 4.250666 7.666451               1.555706  0.249639
      forest_base_10_10     base 3.917248 4.253645 7.652689               1.566750  0.252330
     forest_base_None_2     base 3.985730 4.216904 7.560798               1.660986  0.270178
    forest_base_None_10     base 3.912801 4.202698 7.575046           

In [3]:
for group, rows in result['insights'].items():
    print('Scope:', group)
    print(pd.DataFrame(rows).to_string(index=False))
print('Under-supported groups:', sum(not r['supported'] for r in result['breakdown']))
print('Follow-up: inspect backfills and season definitions; differences are not success or causal effects.')


Scope: all
content_type   field           name  support  reference_support        reference  median_years  reference_median  difference  supported direction                                                                                              limitation
       Movie   genre Classic Movies      116               6119 All Movie titles          44.0               2.0        42.0       True    higher Snapshot association only; overlapping labels and catalog survivorship prevent causal or demand claims.
       Movie   genre    Cult Movies       70               6119 All Movie titles          25.0               2.0        23.0       True    higher Snapshot association only; overlapping labels and catalog survivorship prevent causal or demand claims.
       Movie country      Hong Kong       75               6119 All Movie titles          12.0               2.0        10.0       True    higher Snapshot association only; overlapping labels and catalog survivorship prevent causal or dema

## Figures and limitations

![Eligible calendar-year differences](../../reports/task-06/figures/distribution.png)

![Chronological test predictions](../../reports/task-06/figures/test-predictions.png)

The source is a retrospective catalog, not a point-in-time acquisition history. Removed titles and later backfills are absent. TV release year may refer to a season. 2021 coverage is partial. Multi-label countries and genres overlap; n≥30 is only a support threshold. Post-test permutation importance diagnoses the frozen model and cannot establish causality. Hypotheses require further verification and outcome data before business action. No current forecast or uncertainty interval is claimed.
